In [2]:
from icecube import dataclasses, dataio, icetray
import numpy as np
from glob import glob
import matplotlib.pyplot as plt
import time
import matplotlib.ticker as ticker
from matplotlib.ticker import ScalarFormatter
from icecube.phys_services import I3Calculator

In [1]:
params = {'figure.figsize': (7, 7*0.618),
          'legend.fontsize': 14,
          'axes.labelsize': 16,
          'axes.titlesize': 16,
          'xtick.labelsize': 16,
          'ytick.labelsize': 16}
plt.rcParams.update(params)


KeyboardInterrupt



In [3]:
filelist = sorted(glob('/data/user/akatil/electron_neutrino/for_real/dataset_complete/enriched_029_median_iqr_skew_kurtosis_beta/161029/upgrade_genie_level4_queso_*.i3.zst'))

In [4]:
#getting the geometry frame. We want this to calculate the mean of upgrade om x and y positions
gcd_infile = dataio.I3File('/home/akatil/GeoCalibDetectorStatus_ICUpgrade.v58.mixed.V1.i3.bz2')

f_geo = gcd_infile
geo_frame = f_geo.pop_frame(icetray.I3Frame.Geometry)
geo = geo_frame['I3Geometry']

count = 0

nan_count = 0
for file in filelist:
    count += 1
    if count % 50 == 0:
        print(count)
    infile=dataio.I3File(file)
    while(infile.more()):
        frame = infile.pop_frame()
        if frame.Stop == icetray.I3Frame.Physics:
            
            mctree = frame["I3MCTree"]
            primary = mctree.primaries

            daughter = dataclasses.I3MCTree.first_child(mctree, primary[0].id)
            
            if np.isnan(frame['Beta1Reco'].value):
                #print('Oh no this problem persists')

                nan_count += 1

                pulses = frame["SplitInIcePulses_dynedge_v2_Pulses"]
                hits = pulses.apply(frame)

                for entry in hits:
                    if geo.omgeo.get(entry.key()).omtype.name == 'mDOM':
                        for hit in entry.data():
                            if len(entry.data()) < 1 or len(entry.data()) > 1:
                                print(f'something fishy is going on here. Num hits is {len(entry.data())} and the count is {count} and the file is {file}')
                        

                
print(f'Total number of problem events: {nan_count}')

50
100
150
200
250
300
350
400
450
500
550
600
650
Total number of problem events: 609


In [5]:
daughter.shape

icecube._dataclasses.ParticleShape.Cascade

In [5]:
filelist = sorted(glob('/data/user/akatil/electron_neutrino/for_real/dataset_complete/NuE_BDT_stage/140029/upgrade_genie_level4_queso_*.i3.zst'))

#getting the geometry frame. We want this to calculate the mean of upgrade om x and y positions
gcd_infile = dataio.I3File('/home/akatil/GeoCalibDetectorStatus_ICUpgrade.v58.mixed.V1.i3.bz2')

f_geo = gcd_infile
geo_frame = f_geo.pop_frame(icetray.I3Frame.Geometry)
geo = geo_frame['I3Geometry']

count = 0
for file in filelist[1003:1004]:
    count += 1
    if count % 50 == 0:
        print(count)
    infile=dataio.I3File(file)
    while(infile.more()):
        frame = infile.pop_frame()
        if frame.Stop == icetray.I3Frame.Physics:
            
            mctree = frame["I3MCTree"]
            primary = mctree.primaries

            daughter = dataclasses.I3MCTree.first_child(mctree, primary[0].id)

                    #pid value
            pid = frame['graphnet_dynedge_track_classification_track_pred'].value
        
            #reconstructed position
            rx, ry, rz = frame['graphnet_dynedge_position_reconstruction_position_x_pred'].value, frame['graphnet_dynedge_position_reconstruction_position_y_pred'].value, frame['graphnet_dynedge_position_reconstruction_position_z_pred'].value
        
            #reconstructed time
            reco_vertex_time = frame["reco_vertex_time"].value
            
            reco_energy = frame["graphnet_dynedge_energy_reconstruction_energy_pred"].value
        
            #reconstructed zenith
            reco_dir_x = frame['graphnet_dynedge_direction_reconstruction_dir_x_pred'].value
            reco_dir_y = frame['graphnet_dynedge_direction_reconstruction_dir_y_pred'].value
            reco_dir_z = frame['graphnet_dynedge_direction_reconstruction_dir_z_pred'].value
        
            #All the pulses in the event
            pulses = frame["SplitInIcePulses_dynedge_v2_Pulses"]
            hits = pulses.apply(frame)
            
            ex, ey, ez = -reco_dir_x, -reco_dir_y, -reco_dir_z
        
            daughter.pos = dataclasses.I3Position(rx, ry, rz)
            daughter.time = reco_vertex_time

            original_shape = daughter.shape
            original_dir = daughter.dir
             
            print('og shape', original_shape)

            om_prop = []
            for entry in hits:
                if geo.omgeo.get(entry.key()).omtype.name == 'mDOM':
                    #print('hit length', len(entry.data())) 
                    #position on the om
                    omgeo_pos = geo.omgeo.get(entry.key()).position
                    ox, oy, oz = omgeo_pos
        
                    #Get Angular Distribution
                    #vector b/w particle and OM
                    hx = daughter.pos.x - ox
                    hy = daughter.pos.y - oy
                    hz = daughter.pos.z - oz
        
                    #dot product
                    s = ex*hx + ey*hy + ez*hz
        
                    angle = -s/(np.sqrt(hx**2+hy**2+hz**2))#np.arccos(-s/(np.sqrt(hx**2+hy**2+hz**2)))
        
                    omgeo_pos = geo.omgeo.get(entry.key()).position
                    
                    d = np.sqrt((daughter.pos.x-ox)**2+(daughter.pos.y-oy)**2+(daughter.pos.z-oz)**2)
        
                    for hit in entry.data():

                        daughter.shape = original_shape

                        #print(daughter.shape)
        
                        #Get Time Residuals
                        time_residual = I3Calculator.time_residual(daughter, omgeo_pos,
                                                                         hit.time)

                        
                        daughter.shape = dataclasses.I3Particle.StartingTrack

                        #print(daughter.shape)

                        time_residual_change_shape = I3Calculator.time_residual(daughter, omgeo_pos,
                                                                         hit.time)

                        print(time_residual-time_residual_change_shape)
                        
                        

                        #if daughter.shape != dataclasses.I3Particle.Cascade: #if np.isnan(time_residual):
                            #print('This is nan', daughter.shape, time_residual)
                            #print(daughter.shape, time_residual)


                    
                    

og shape Dark
nan
nan
nan
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
og shape Dark
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
og shape Dark
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
0.0
0.0
nan
nan
nan
nan
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
og shape Dark
nan
nan
nan
nan
nan
nan
nan
nan
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
nan
nan
0.0
0.0
0.0
0.0
0.0
0.0
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
0.0
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
og shape Dark
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
nan
na

In [17]:
filelist = sorted(glob('/data/user/akatil/electron_neutrino/for_real/dataset_complete/NuE_BDT_stage/16*29/upgrade_genie_level4_queso_*.i3.zst'))

In [18]:
len(filelist)

2199

In [ ]:


count = 0
for file in filelist:
    count += 1
    if count % 50 == 0:
        print(count)
    infile=dataio.I3File(file)
    while(infile.more()):
        frame = infile.pop_frame()
        if frame.Stop == icetray.I3Frame.Physics:
            
        

In [5]:
import os

filelist = []

folder_path = "../../../dataset_complete/variables/muon_neutrino/"

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        filelist.append(file_path)  # do whatever you want with file_path

In [6]:
filelist[0:100]

['../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000558.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_00058.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000544.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000561.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000567.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000565.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000598.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141829_00000.h5',
 '../../../dataset_complete/variables/muon_neutrino/batch_307/upgrade_genie_level4_queso_141828_000557.h5',
 '../../../dataset_complete/va